In [3]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/yashpaloswal/ann-car-sales-price-prediction/car_purchasing.csv


In [5]:
import pandas as pd

df = pd.read_csv(
    "../input/datasets/yashpaloswal/ann-car-sales-price-prediction/car_purchasing.csv",
    encoding="latin1"
)

df.head()

,customer name,customer e-mail,country,gender,age,annual Salary,credit card debt,net worth,car purchase amount
0,Martina Avila,cubilia.Curae.Phasellus@quisaccumsanconvallis.edu,Bulgaria,0,41.851720,62812.09301,11609.380910,238961.2505,35321.45877
1,Harlan Barnes,eu.dolor@diam.co.uk,Belize,0,40.870623,66646.89292,9572.957136,530973.9078,45115.52566
2,Naomi Rodriquez,vulputate.mauris.sagittis@ametconsectetueradip...,Algeria,1,43.152897,53798.55112,11160.355060,638467.1773,42925.70921
3,Jade Cunningham,malesuada@dignissim.com,Cook Islands,1,58.271369,79370.03798,14426.164850,548599.0524,67422.36313
4,Cedric Leach,felis.ullamcorper.viverra@egetmollislectus.net,Brazil,1,57.313749,59729.15130,5358.712177,560304.0671,55915.46248


In [6]:
df.describe()

,gender,age,annual Salary,credit card debt,net worth,car purchase amount
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,0.506000,46.241674,62127.239608,9607.645049,431475.713625,44209.799218
std,0.500465,7.978862,11703.378228,3489.187973,173536.756340,10773.178744
min,0.000000,20.000000,20000.000000,100.000000,20000.000000,9000.000000
25%,0.000000,40.949969,54391.977195,7397.515792,299824.195900,37629.896040
50%,1.000000,46.049901,62915.497035,9655.035568,426750.120650,43997.783390
75%,1.000000,51.612263,70117.862005,11798.867487,557324.478725,51254.709517
max,1.000000,70.000000,100000.000000,20000.000000,1000000.000000,80000.000000


In [7]:
print(df.select_dtypes(include='object').columns)

Index(['customer name', 'customer e-mail', 'country'], dtype='object')


In [8]:
df.columns.tolist()

['customer name',
 'customer e-mail',
 'country',
 'gender',
 'age',
 'annual Salary',
 'credit card debt',
 'net worth',
 'car purchase amount']

In [9]:
df.drop(['customer name', 'customer e-mail', 'country'], axis=1, inplace=True)

In [10]:
X = df.drop('car purchase amount', axis=1)
y = df['car purchase amount']

In [11]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

In [12]:
from sklearn.preprocessing import StandardScaler

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train = x_scaler.fit_transform(X_train)
X_val = x_scaler.transform(X_val)
X_test = x_scaler.transform(X_test)

y_train = y_scaler.fit_transform(y_train.values.reshape(-1,1))
y_val = y_scaler.transform(y_val.values.reshape(-1,1))
y_test = y_scaler.transform(y_test.values.reshape(-1,1))


In [13]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [14]:
dtype=torch.float32

In [15]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)
        
    def forward(self, x):
        return self.fc(x)

model = MyModel()

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=0.001
)

In [16]:
from torch.utils.data import Dataset

class CarDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [17]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    CarDataset(X_train, y_train),
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    CarDataset(X_val, y_val),
    batch_size=16
)

test_loader = DataLoader(
    CarDataset(X_test, y_test),
    batch_size=16
)

In [18]:
import torch.nn as nn

class CarANN(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)   # ONE output
        )

    def forward(self, x):
        return self.net(x)

In [19]:
model = CarANN(X_train.shape[1])

In [20]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [21]:
epochs = 100

for epoch in range(epochs):

    model.train()

    train_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = model(X_batch)

        loss = criterion(predictions, y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {train_loss:.4f}"
        )

Epoch [10/100] Loss: 0.4819
Epoch [20/100] Loss: 0.0204
Epoch [30/100] Loss: 0.0055
Epoch [40/100] Loss: 0.0032
Epoch [50/100] Loss: 0.0025
Epoch [60/100] Loss: 0.0020
Epoch [70/100] Loss: 0.0018
Epoch [80/100] Loss: 0.0014
Epoch [90/100] Loss: 0.0012
Epoch [100/100] Loss: 0.0010


In [22]:
epochs = 100

for epoch in range(epochs):

    # Training
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = model(X_batch)

        loss = criterion(predictions, y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            predictions = model(X_batch)

            loss = criterion(predictions, y_batch)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f}"
        )

Epoch [10/100] | Train Loss: 0.0009 | Val Loss: 0.0008
Epoch [20/100] | Train Loss: 0.0008 | Val Loss: 0.0008
Epoch [30/100] | Train Loss: 0.0007 | Val Loss: 0.0009
Epoch [40/100] | Train Loss: 0.0006 | Val Loss: 0.0007
Epoch [50/100] | Train Loss: 0.0006 | Val Loss: 0.0007
Epoch [60/100] | Train Loss: 0.0005 | Val Loss: 0.0005
Epoch [70/100] | Train Loss: 0.0005 | Val Loss: 0.0005
Epoch [80/100] | Train Loss: 0.0004 | Val Loss: 0.0004
Epoch [90/100] | Train Loss: 0.0003 | Val Loss: 0.0003
Epoch [100/100] | Train Loss: 0.0003 | Val Loss: 0.0003


In [23]:
model.eval()

with torch.no_grad():

    predictions = model(X_test)

    mse = criterion(predictions, y_test)

print("Test MSE:", mse.item())

Test MSE: 0.0004303654422983527


In [24]:
from sklearn.metrics import r2_score

y_pred = predictions.numpy()
y_true = y_test.numpy()

print("R² Score:", r2_score(y_true, y_pred))

R² Score: 0.9995322823524475


In [25]:
print(df.columns.tolist())

['gender', 'age', 'annual Salary', 'credit card debt', 'net worth', 'car purchase amount']


In [26]:
from sklearn.metrics import r2_score

model.eval()

with torch.no_grad():
    train_pred = model(X_train).numpy()
    val_pred = model(X_val).numpy()
    test_pred = model(X_test).numpy()

print("Train R²:", r2_score(y_train.numpy(), train_pred))
print("Val R²:", r2_score(y_val.numpy(), val_pred))
print("Test R²:", r2_score(y_test.numpy(), test_pred))

Train R²: 0.9997109770774841
Val R²: 0.9996336698532104
Test R²: 0.9995322823524475
